# Multi-Label Eye Disease Classification 

## Setup

In [ ]:

import os
import re
import pickle
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
import torchmetrics
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore", category=UserWarning, module="torchmetrics")
warnings.filterwarnings("ignore", category=UserWarning, module="albumentations")
warnings.filterwarnings("ignore", category=UserWarning, module="timm")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Using device: {DEVICE}")


## Stage 1 — Data-Level Fusion

ODIR-5K ships one row per **patient** (with separate left/right eye columns and free-text
diagnostic keywords), while RFMiD ships one row per **image** with binary disease columns.
We normalise both into a common `(patient_id, image_path, <disease columns>)` schema, concatenate
them, and perform a **patient-aware** split so that both eyes of the same patient always land in
the same split (avoiding data leakage between train/val/test).

In [ ]:

# Raw dataset locations 
ODIR_PATH = '/kaggle/input/odir-5knew-dataset'
RFMID_PATH = '/kaggle/input/rfmid-fundus-images'

FUSED_CSV_PATH = '/kaggle/working/final_processed_dataset.csv'

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

def process_odir():
    
    odir_excel = pd.read_excel(f'{ODIR_PATH}/ODIR-5K/ODIR-5K/data.xlsx')
    patient_id_col = 'ID'  

    left_df = pd.DataFrame({
        'patient_id': odir_excel[patient_id_col],
        'image_path': f'{ODIR_PATH}/ODIR-5K/ODIR-5K/Training Images/' + odir_excel['Left-Fundus'],
        'keywords': odir_excel['Left-Diagnostic Keywords'],
    })
    right_df = pd.DataFrame({
        'patient_id': odir_excel[patient_id_col],
        'image_path': f'{ODIR_PATH}/ODIR-5K/ODIR-5K/Training Images/' + odir_excel['Right-Fundus'],
        'keywords': odir_excel['Right-Diagnostic Keywords'],
    })

    odir_df = pd.concat([left_df, right_df], ignore_index=True).dropna()

    labels = odir_df['keywords'].str.get_dummies(sep='，')
    odir_df['N'] = labels.get('normal fundus', 0)
    odir_df['D'] = labels.get('diabetic retinopathy', 0)
    odir_df['G'] = labels.get('glaucoma', 0)
    odir_df['C'] = labels.get('cataract', 0)
    odir_df['A'] = labels.get('age-related macular degeneration', 0)
    odir_df['H'] = labels.get('hypertensive retinopathy', 0)
    odir_df['M'] = labels.get('pathological myopia', 0)
    odir_df['O'] = labels.get('other diseases/abnormalities', 0)
    odir_df.drop(columns=['keywords'], inplace=True)

    print(f" -> {len(odir_df)} images")
    return odir_df


def process_rfmid():
    
    print("Processing RFMiD dataset...")
    rfmid_df = pd.read_csv(f'{RFMID_PATH}/Training_Set/Training_Set/RFMiD_Training_Labels.csv')
    rfmid_df.rename(columns={'ID': 'image_id', 'Disease_Risk': 'has_risk'}, inplace=True)
    rfmid_df['image_path'] = f'{RFMID_PATH}Training_Set/' + rfmid_df['image_id'].astype(str) + '.png'
    rfmid_df['patient_id'] = 'rfmid_' + rfmid_df['image_id'].astype(str)

    print(f"  -> {len(rfmid_df)} images")
    return rfmid_df


def fuse_and_split():
    odir_data = process_odir()
    rfmid_data = process_rfmid()

    print("\nUnifying labels across datasets...")
    label_mapping = {
        'N': 'Normal', 'D': 'DR', 'A': 'AMD', 'H': 'HR',
        'G': 'GLAU', 'C': 'CATARACT', 'M': 'MYA', 'O': 'OTH',
    }
    odir_data.rename(columns=label_mapping, inplace=True)

    non_label_cols = {'patient_id', 'image_path', 'image_id', 'has_risk'}
    all_labels = sorted((set(odir_data.columns) | set(rfmid_data.columns)) - non_label_cols)
    print(f"Total unique labels: {len(all_labels)} -> {all_labels}")

    combined_df = pd.concat([odir_data, rfmid_data], ignore_index=True)
    combined_df[all_labels] = combined_df[all_labels].fillna(0).astype(int)
    final_df = combined_df[['patient_id', 'image_path'] + all_labels]

    print(f"\nCombination complete: {len(final_df)} images, "
          f"{final_df['patient_id'].nunique()} unique patients.")

    # Patient-aware split so both eyes of a patient stay together 
    patient_ids = final_df['patient_id'].unique()
    rng = np.random.default_rng(SEED)
    rng.shuffle(patient_ids)

    train_end = int(len(patient_ids) * TRAIN_RATIO)
    val_end = int(len(patient_ids) * (TRAIN_RATIO + VAL_RATIO))
    train_ids, val_ids = set(patient_ids[:train_end]), set(patient_ids[train_end:val_end])

    def assign_split(pid):
        if pid in train_ids:
            return 'train'
        if pid in val_ids:
            return 'val'
        return 'test'

    final_df['split'] = final_df['patient_id'].apply(assign_split)
    print("\nImage distribution across splits:")
    print(final_df['split'].value_counts())

    final_df.to_csv(FUSED_CSV_PATH, index=False)
    print(f"\nSaved fused dataset to {FUSED_CSV_PATH}")
    return final_df, all_labels


fused_df, all_labels = fuse_and_split()


In [ ]:

def archive_dataset_for_reuse(csv_path=FUSED_CSV_PATH,
                               odir_root=ODIR_PATH,
                               rfmid_root=RFMID_PATH,
                               output_zip='/kaggle/working/combined_eyecare_dataset.zip'):
    
    import zipfile

    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_path, arcname=os.path.basename(csv_path))
        for root_path in (odir_root, rfmid_root):
            for root, _, files in os.walk(root_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    archive_path = os.path.relpath(file_path, '/kaggle/input/')
                    zipf.write(file_path, arcname=archive_path)

    print(f"Zip archive created at: {output_zip}")


## Stage 2 - Data-Driven Problem Enhancement & Image Preprocessing
We inspect class frequency and **focus the problem** on the 6
classes with enough samples: `Normal, DR, MYA, MH, CATARACT, ODC`. We then build the `Focused Dataset` 

In [ ]:

def analyse_class_frequency(csv_path=FUSED_CSV_PATH):
    
    df = pd.read_csv(csv_path)
    label_columns = df.columns.drop(['image_path', 'patient_id', 'split'])
    disease_counts = df[label_columns].sum().sort_values(ascending=False)

    print("Most frequent diseases in the combined dataset")
    print(disease_counts.to_string())
    return disease_counts


TARGET_CLASSES = ['Normal', 'DR', 'MYA', 'MH', 'CATARACT', 'ODC']
NUM_CLASSES = len(TARGET_CLASSES)

FOCUSED_CSV_PATH = '/kaggle/working/focused_dataset.csv'


def build_focused_dataset(csv_path=FUSED_CSV_PATH, target_classes=TARGET_CLASSES,
                          output_path=FOCUSED_CSV_PATH):
   
    df = pd.read_csv(csv_path)
    df['filename'] = df['image_path'].apply(os.path.basename)

    keep_cols = ['filename', 'image_path', 'patient_id', 'split'] + target_classes
    focused_df = df[keep_cols].copy()
    focused_df = focused_df[focused_df[target_classes].sum(axis=1) > 0].reset_index(drop=True)

    focused_df.to_csv(output_path, index=False)
    print(f"Focused dataset: {len(focused_df)} images across {target_classes}")
    print(f"Saved to {output_path}")
    return focused_df


def plot_class_distribution(csv_path=FOCUSED_CSV_PATH, target_classes=TARGET_CLASSES,
                             output_path='/kaggle/working/class_distribution_plot.png'):
    df = pd.read_csv(csv_path)
    class_counts = df[target_classes].sum()
    plot_df = pd.DataFrame({'Class': class_counts.index, 'Image Count': class_counts.values})
    plot_df = plot_df.sort_values(by='Image Count', ascending=False)

    plt.figure(figsize=(12, 8))
    sns.set_style("whitegrid")
    ax = sns.barplot(x='Class', y='Image Count', data=plot_df, palette='viridis')
    ax.set_title('Class Distribution in the Final Training Dataset', fontsize=18)
    ax.set_xlabel('Disease Class', fontsize=14)
    ax.set_ylabel('Number of Images', fontsize=14)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontsize=12)
    plt.tight_layout()
    plt.savefig(output_path)
    print(f"Saved class distribution plot to {output_path}")
    plt.show()


analyse_class_frequency()
build_focused_dataset()
plot_class_distribution()


In [ ]:

#Image source directories (used to reconstruct paths from a bare filename)
ODIR_IMAGES_PATH = '/kaggle/input/odir-5knew-dataset/ODIR-5K/ODIR-5K/Training Images'
RFMID_IMAGES_PATH = '/kaggle/input/rfmid-fundus-images/Training_Set/Training_Set/Training'

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2


def reconstruct_image_path(filename):
   
    filename = str(filename).strip()
    if '_left.jpg' in filename or '_right.jpg' in filename:
        return os.path.join(ODIR_IMAGES_PATH, filename)
    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        filename += '.png'
    return os.path.join(RFMID_IMAGES_PATH, filename)


def load_focused_dataframe(csv_path=FOCUSED_CSV_PATH):
    
    df = pd.read_csv(csv_path)
    if 'filename' not in df.columns:
        df['filename'] = df['image_path'].apply(os.path.basename)
    df['image_path'] = df['filename'].apply(reconstruct_image_path)
    return df


class EyeDataset(Dataset):


    def __init__(self, df, transforms=None, target_classes=TARGET_CLASSES):
        self.df = df
        self.image_paths = df['image_path'].values
        self.labels = df[target_classes].values
        self.transforms = transforms
        self.num_classes = len(target_classes)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        try:
            image = cv2.imread(image_path)
            if image is None:
                raise FileNotFoundError(f"cv2.imread failed for {image_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except Exception as e:
            print(f"ERROR reading {image_path}: {e}")
            return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(self.num_classes)

        if self.transforms:
            image = self.transforms(image=image)['image']

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label


def get_transforms(img_size=IMG_SIZE):
    
    train_transforms = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    eval_transforms = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return train_transforms, eval_transforms


## Stage 3 — Training
We train **Swin-Tiny** (`swin_tiny_patch4_window7_224`) as the primary model.

In [ ]:

EPOCHS = 15
LEARNING_RATE = 1e-5

PRIMARY_MODEL = 'swin_tiny_patch4_window7_224'
COMPARISON_MODELS = ['resnet50', 'efficientnet_b3', 'densenet121', 'vit_base_patch16_224']


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader)


def validate(model, loader, loss_fn, device, metrics_calculator):
    model.eval()
    total_loss = 0
    metrics_calculator.reset()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            metrics_calculator.update(outputs, labels.int())
    metrics = metrics_calculator.compute()
    return total_loss / len(loader), metrics


def train_model(model_name, img_size=IMG_SIZE, epochs=EPOCHS, lr=LEARNING_RATE,
                focused_csv_path=FOCUSED_CSV_PATH):
    
    print(f"\n Training {model_name}")
    df = load_focused_dataframe(focused_csv_path)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df = df[df['split'] == 'val'].reset_index(drop=True)

    train_transforms, val_transforms = get_transforms(img_size)
    train_loader = DataLoader(EyeDataset(train_df, train_transforms), batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(EyeDataset(val_df, val_transforms), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS)

    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

    pos_counts = train_df[TARGET_CLASSES].sum().values
    neg_counts = len(train_df) - pos_counts
    pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    metrics_calculator = torchmetrics.MetricCollection({
        'accuracy': torchmetrics.Accuracy(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'precision': torchmetrics.Precision(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'recall': torchmetrics.Recall(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'f1': torchmetrics.F1Score(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'auroc': torchmetrics.AUROC(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
    }).to(DEVICE)

    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_precision': [],
               'val_recall': [], 'val_f1': [], 'val_auroc': []}

    best_f1, ckpt_path = 0, f"{model_name}_best_model.pth"
    for epoch in range(epochs):
        print(f"--- Epoch {epoch + 1}/{epochs} ---")
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss, metrics = validate(model, val_loader, loss_fn, DEVICE, metrics_calculator)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val F1: {metrics['f1']:.4f} | Val AUROC: {metrics['auroc']:.4f}")

        for key in history:
            metric_key = key.replace('val_', '')
            history[key].append(train_loss if key == 'train_loss'
                                 else val_loss if key == 'val_loss'
                                 else metrics[metric_key].item())

        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            torch.save(model.state_dict(), ckpt_path)
            print(f"  New best F1 ({best_f1:.4f}) -> saved {ckpt_path}")

    with open(f"{model_name}_history.pkl", 'wb') as f:
        pickle.dump(history, f)

    return history

primary_history = train_model(PRIMARY_MODEL)
comparison_histories = {name: train_model(name) for name in COMPARISON_MODELS}


In [ ]:

def plot_training_history(history, model_name, output_path=None):
    history_df = pd.DataFrame(history)
    history_df['epoch'] = history_df.index + 1

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f'{model_name} — Training & Validation History', fontsize=20, y=1.03)

    sns.lineplot(x='epoch', y='train_loss', data=history_df, ax=axes[0, 0], label='Train Loss', marker='o')
    sns.lineplot(x='epoch', y='val_loss', data=history_df, ax=axes[0, 0], label='Validation Loss', marker='o')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend()

    sns.lineplot(x='epoch', y='val_f1', data=history_df, ax=axes[0, 1], label='Validation F1', marker='o', color='green')
    axes[0, 1].set_title('F1 Score')

    sns.lineplot(x='epoch', y='val_precision', data=history_df, ax=axes[1, 0], label='Precision', marker='o')
    sns.lineplot(x='epoch', y='val_recall', data=history_df, ax=axes[1, 0], label='Recall', marker='o')
    axes[1, 0].set_title('Precision & Recall'); axes[1, 0].legend()

    sns.lineplot(x='epoch', y='val_accuracy', data=history_df, ax=axes[1, 1], label='Accuracy', marker='o', color='cyan')
    axes[1, 1].set_title('Accuracy')

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
        print(f"Saved training history plot to {output_path}")
    plt.show()


def evaluate_model_on_test_set(model_name, img_size=IMG_SIZE, focused_csv_path=FOCUSED_CSV_PATH,
                                checkpoint_path=None):
    
    checkpoint_path = checkpoint_path or f"{model_name}_best_model.pth"
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    df = load_focused_dataframe(focused_csv_path)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    _, eval_transforms = get_transforms(img_size)
    test_loader = DataLoader(EyeDataset(test_df, eval_transforms), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS)

    per_class_metrics = torchmetrics.MetricCollection({
        'Accuracy': torchmetrics.Accuracy(task="multilabel", num_labels=NUM_CLASSES, average='none'),
        'Precision': torchmetrics.Precision(task="multilabel", num_labels=NUM_CLASSES, average='none'),
        'Recall': torchmetrics.Recall(task="multilabel", num_labels=NUM_CLASSES, average='none'),
        'F1Score': torchmetrics.F1Score(task="multilabel", num_labels=NUM_CLASSES, average='none'),
        'AUROC': torchmetrics.AUROC(task="multilabel", num_labels=NUM_CLASSES, average='none'),
    }).to(DEVICE)
    overall_metrics = torchmetrics.MetricCollection({
        'Accuracy': torchmetrics.Accuracy(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'Precision': torchmetrics.Precision(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'Recall': torchmetrics.Recall(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'F1Score': torchmetrics.F1Score(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
        'AUROC': torchmetrics.AUROC(task="multilabel", num_labels=NUM_CLASSES, average='macro'),
    }).to(DEVICE)

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Evaluating {model_name}"):
            images, labels = images.to(DEVICE), labels.int().to(DEVICE)
            outputs = model(images)
            per_class_metrics.update(outputs, labels)
            overall_metrics.update(outputs, labels)

    per_class_results = per_class_metrics.compute()
    per_class_df = pd.DataFrame({'Class': TARGET_CLASSES, **{
        k: per_class_results[k].cpu().numpy() for k in
        ['Accuracy', 'Precision', 'Recall', 'F1Score', 'AUROC']
    }})
    print(f"\nPer-Class Metrics ({model_name})")
    print(per_class_df.round(4).to_string(index=False))

    overall_results = overall_metrics.compute()
    overall_df = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUROC'],
        'Value': [overall_results[k].item() for k in
                  ['Accuracy', 'Precision', 'Recall', 'F1Score', 'AUROC']],
    })
    print(f"\n--- Overall Metrics ({model_name}) ---")
    print(overall_df.round(4).to_string(index=False))

    return per_class_df, overall_df


plot_training_history(primary_history, PRIMARY_MODEL, '/kaggle/working/training_history_plot.png')
swin_tiny_per_class, swin_tiny_overall = evaluate_model_on_test_set(PRIMARY_MODEL)
comparison_results = {name: evaluate_model_on_test_set(name) for name in COMPARISON_MODELS}


In [ ]:

def show_sample_predictions(model_name=PRIMARY_MODEL, img_size=IMG_SIZE,
                             focused_csv_path=FOCUSED_CSV_PATH, checkpoint_path=None,
                             tune_ratio=0.8, n_examples=5, required_true_label_count=2):
    
    checkpoint_path = checkpoint_path or f"{model_name}_best_model.pth"
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    df = load_focused_dataframe(focused_csv_path)
    test_df_full = df[df['split'] == 'test'].reset_index(drop=True)
    _, eval_transforms = get_transforms(img_size)

    tune_df, _ = train_test_split(test_df_full, test_size=1 - tune_ratio,
                                   random_state=SEED, stratify=test_df_full['DR'])

    def get_predictions(loader):
        preds, labels = [], []
        with torch.no_grad():
            for images, batch_labels in tqdm(loader, desc="Predicting", leave=False):
                images = images.to(DEVICE)
                outputs = model(images)
                preds.append(torch.sigmoid(outputs).cpu())
                labels.append(batch_labels.cpu())
        return torch.cat(preds), torch.cat(labels).int()

    tune_loader = DataLoader(EyeDataset(tune_df, eval_transforms), batch_size=BATCH_SIZE, shuffle=False)
    tune_preds, tune_labels = get_predictions(tune_loader)

    optimal_thresholds = {}
    for i, class_name in enumerate(TARGET_CLASSES):
        y_true, y_prob = tune_labels[:, i].numpy(), tune_preds[:, i].numpy()
        if len(np.unique(y_true)) < 2:
            optimal_thresholds[class_name] = 0.5
            continue
        best_f1, best_thresh = -1, 0.5
        for thresh in np.arange(0.01, 1.0, 0.01):
            f1 = f1_score(y_true, (y_prob > thresh).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh
        optimal_thresholds[class_name] = best_thresh

    full_loader = DataLoader(EyeDataset(test_df_full, eval_transforms), batch_size=BATCH_SIZE, shuffle=False)
    all_preds, all_labels = get_predictions(full_loader)

    all_preds_binary = torch.zeros_like(all_labels)
    for i, class_name in enumerate(TARGET_CLASSES):
        all_preds_binary[:, i] = (all_preds[:, i] > optimal_thresholds[class_name]).int()

    perfect_matches = torch.all(all_preds_binary == all_labels, dim=1)
    n_true_labels = all_labels.sum(dim=1)
    target_mask = perfect_matches & (n_true_labels == required_true_label_count)
    target_indices = np.where(target_mask.numpy())[0][:n_examples]

    print(f"\n{len(target_indices)} example(s) with a perfect, "
          f"{required_true_label_count}-label prediction")
    results = []
    for i in target_indices:
        true_labels = [TARGET_CLASSES[j] for j, v in enumerate(all_labels[i]) if v == 1]
        pred_labels = [TARGET_CLASSES[j] for j, v in enumerate(all_preds_binary[i]) if v == 1]
        results.append({
            'Image File': test_df_full['filename'].iloc[i],
            'True Labels': ', '.join(true_labels) or 'Normal',
            'Predicted Labels': ', '.join(pred_labels) or 'Normal',
        })
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    return results_df

show_sample_predictions()


## Stage 4 — Explainable AI (Grad-CAM)

We integrated **Grad-CAM** for clinical explainability and reliability.

In [ ]:

# pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


# A few test images chosen to showcase single-label, multi-label, and edge cases.
GRADCAM_TARGET_FILENAMES = ['78.png', '608.png', '842.png', '1122.png', '1403.png', '1453.png']


class GradCAMDataset(Dataset):
    '''Like EyeDataset, but also returns the untransformed (resized) RGB image so
    it can be overlaid with the CAM heatmap.'''

    def __init__(self, df, transforms, img_size=IMG_SIZE, target_classes=TARGET_CLASSES):
        self.df = df
        self.image_paths = df['image_path'].values
        self.labels = df[target_classes].values
        self.transforms = transforms
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        original_image = cv2.imread(image_path)
        if original_image is None:
            print(f"ERROR: could not read {image_path}. Skipping.")
            return None, None, None
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        original_image = cv2.resize(original_image, (self.img_size, self.img_size))

        transformed_image = self.transforms(image=original_image)['image']
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return original_image, transformed_image, label


def get_gradcam_transforms():
    return A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def generate_gradcam_explanations(model_name=PRIMARY_MODEL, checkpoint_path=None,
                                   focused_csv_path=FOCUSED_CSV_PATH,
                                   target_filenames=GRADCAM_TARGET_FILENAMES,
                                   output_dir='.'):
    '''Generate one Grad-CAM figure per predicted class for each target image.'''
    checkpoint_path = checkpoint_path or f"{model_name}_best_model.pth"
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE).eval()

    def reshape_transform(tensor, height=7, width=7):
        '''Swin's output is (batch, tokens, channels); reshape back to a spatial
        (batch, channels, height, width) feature map for Grad-CAM.'''
        result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
        return result.transpose(2, 3).transpose(1, 2)

    # Final normalisation layer of the last Swin block.
    target_layer = model.layers[-1].blocks[-1].norm2

    df = load_focused_dataframe(focused_csv_path)
    target_df = df[df['filename'].isin(target_filenames)].reset_index(drop=True)
    if len(target_df) == 0:
        print("None of the target filenames were found in the dataset.")
        return
    print(f"Found {len(target_df)} of the requested images. Generating explanations...")

    dataset = GradCAMDataset(target_df, transforms=get_gradcam_transforms())

    for i in range(len(dataset)):
        original_image_np, transformed_image, label_tensor = dataset[i]
        if original_image_np is None:
            continue

        input_tensor = transformed_image.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prediction = model(input_tensor)
            pred_indices = torch.where(torch.sigmoid(prediction).squeeze() > 0.5)[0]

        filename = target_df['filename'].iloc[i]
        if len(pred_indices) == 0:
            print(f"{filename}: model predicted 'None'. Skipping CAM.")
            continue

        # Re-initialise GradCAM per image for a clean hook state.
        cam = GradCAM(model=model, target_layers=[target_layer], reshape_transform=reshape_transform)
        true_labels = [TARGET_CLASSES[j] for j, v in enumerate(label_tensor) if v == 1]

        for pred_idx in pred_indices:
            target_class_name = TARGET_CLASSES[pred_idx]
            grayscale_cam = cam(input_tensor=input_tensor,
                                 targets=[ClassifierOutputTarget(pred_idx.item())])[0, :]

            vis_image = original_image_np.astype(np.float32) / 255
            visualization = show_cam_on_image(vis_image, grayscale_cam, use_rgb=True)

            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            fig.suptitle(f"Grad-CAM for {filename} | Prediction Focus: {target_class_name} | "
                         f"True Labels: {true_labels}", fontsize=16)
            axes[0].imshow(original_image_np); axes[0].set_title("Original Image"); axes[0].axis('off')
            axes[1].imshow(grayscale_cam, cmap='jet'); axes[1].set_title("Heatmap"); axes[1].axis('off')
            axes[2].imshow(visualization); axes[2].set_title("Overlay"); axes[2].axis('off')
            plt.tight_layout()

            out_path = os.path.join(
                output_dir, f"explanation_{filename.replace('.', '_')}_pred_{target_class_name}.png")
            plt.savefig(out_path)
            plt.show()
            plt.close(fig)
            print(f"Saved: {out_path}")


generate_gradcam_explanations()


## Stage 5 — Cross-Dataset Validation (APTOS 2019)

We checked how well Swin-Tiny generalises to an unseen dataset: **APTOS 2019**,
which only distinguishes *No DR* (grade 0) from *DR* (grades 1–4). We hold out 20% of APTOS to tune a per-class decision threshold (since the model was calibrated on a different label
distribution), then report F1 on the remaining 80%, side-by-side with the in-domain test F1.

In [ ]:

APTOS_DATA_PATH = '/kaggle/input/dr-datasetfinal'  # contains train.csv + train_images/
EVAL_CLASSES = ['Normal', 'DR']  # the only classes APTOS can supply ground truth for
APTOS_TUNE_RATIO = 0.2
DEFAULT_THRESHOLD = 0.5


def collate_skip_none(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty((0, 3, IMG_SIZE, IMG_SIZE)), torch.empty((0, NUM_CLASSES))
    return torch.utils.data.dataloader.default_collate(batch)


def prepare_internal_test_set(focused_csv_path=FOCUSED_CSV_PATH):
    df = load_focused_dataframe(focused_csv_path)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    print(f"Internal (ODIR/RFMiD) test set: {len(test_df)} images.")
    return test_df


def prepare_aptos_dataset(data_path=APTOS_DATA_PATH, target_classes=TARGET_CLASSES):
    '''APTOS only has DR grades, so we map grade 0 -> Normal and grades 1-4 -> DR;
    all other target classes are left at 0 (unknown/absent for this dataset).'''
    df = pd.read_csv(os.path.join(data_path, 'train.csv'))
    df['image_path'] = df['id_code'].apply(lambda x: os.path.join(data_path, 'train_images', f"{x}.png"))
    for col in target_classes:
        df[col] = 0
    df.loc[df['diagnosis'] == 0, 'Normal'] = 1
    df.loc[df['diagnosis'] > 0, 'DR'] = 1
    print(f"Loaded {len(df)} images from APTOS 2019.")
    return df


def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Predicting", leave=False):
            if images.shape[0] == 0:
                continue
            outputs = model(images.to(DEVICE))
            all_preds.append(torch.sigmoid(outputs).cpu())
            all_labels.append(labels.cpu().int())
    return torch.cat(all_preds), torch.cat(all_labels)


def cross_dataset_validation(model_name=PRIMARY_MODEL, checkpoint_path=None,
                              output_path='/kaggle/working/generalization_comparison_adapted_plot.png'):
    '''Compare in-domain F1 (default 0.5 threshold) against out-of-domain APTOS F1
    (threshold re-tuned on a small APTOS holdout), for the classes both datasets share.'''
    checkpoint_path = checkpoint_path or f"{model_name}_best_model.pth"
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE)

    _, eval_transforms = get_transforms(IMG_SIZE)
    plot_rows = []
    normal_idx, dr_idx = TARGET_CLASSES.index('Normal'), TARGET_CLASSES.index('DR')

    #1. In-domain performance at the default threshold 
    internal_df = prepare_internal_test_set()
    internal_loader = DataLoader(EyeDataset(internal_df, eval_transforms), batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_skip_none)
    internal_preds, internal_labels = get_predictions(model, internal_loader)
    internal_preds_binary = (internal_preds > DEFAULT_THRESHOLD).int()
    f1_internal = torchmetrics.F1Score(task="multilabel", num_labels=NUM_CLASSES,
                                        average='none')(internal_preds_binary, internal_labels).numpy()
    plot_rows += [
        {'Dataset': 'Internal (ODIR/RFMiD @ 0.5)', 'Class': 'Normal', 'F1 Score': f1_internal[normal_idx]},
        {'Dataset': 'Internal (ODIR/RFMiD @ 0.5)', 'Class': 'DR', 'F1 Score': f1_internal[dr_idx]},
    ]

    # 2. Out-of-domain performance with a threshold re-tuned on APTOS 
    aptos_df = prepare_aptos_dataset()
    tune_df, test_df = train_test_split(aptos_df, test_size=1 - APTOS_TUNE_RATIO,
                                         random_state=SEED, stratify=aptos_df['DR'])
    print(f"APTOS split: {len(tune_df)} for threshold tuning, {len(test_df)} for final testing.")

    tune_loader = DataLoader(EyeDataset(tune_df, eval_transforms), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_skip_none)
    tune_preds, tune_labels = get_predictions(model, tune_loader)

    optimal_thresholds = {}
    for class_name in EVAL_CLASSES:
        idx = TARGET_CLASSES.index(class_name)
        y_true, y_prob = tune_labels[:, idx].numpy(), tune_preds[:, idx].numpy()
        if len(np.unique(y_true)) < 2:
            optimal_thresholds[class_name] = DEFAULT_THRESHOLD
            continue
        best_f1, best_thresh = -1, DEFAULT_THRESHOLD
        for thresh in np.arange(0.01, 1.0, 0.01):
            f1 = f1_score(y_true, (y_prob > thresh).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh
        optimal_thresholds[class_name] = best_thresh
        print(f"  Optimal threshold for {class_name}: {best_thresh:.2f} (F1={best_f1:.4f})")

    test_loader = DataLoader(EyeDataset(test_df, eval_transforms), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_skip_none)
    test_preds, test_labels = get_predictions(model, test_loader)

    test_preds_binary = (test_preds > DEFAULT_THRESHOLD).int().clone()
    test_preds_binary[:, normal_idx] = (test_preds[:, normal_idx] > optimal_thresholds['Normal']).int()
    test_preds_binary[:, dr_idx] = (test_preds[:, dr_idx] > optimal_thresholds['DR']).int()

    f1_external = torchmetrics.F1Score(task="multilabel", num_labels=NUM_CLASSES,
                                        average='none')(test_preds_binary, test_labels).numpy()
    plot_rows += [
        {'Dataset': 'External (APTOS @ adapted)', 'Class': 'Normal', 'F1 Score': f1_external[normal_idx]},
        {'Dataset': 'External (APTOS @ adapted)', 'Class': 'DR', 'F1 Score': f1_external[dr_idx]},
    ]

    #  3. Plot 
    plot_df = pd.DataFrame(plot_rows)
    print("\n--- Generalisation comparison ---")
    print(plot_df.round(4).to_string(index=False))

    plt.figure(figsize=(10, 7))
    sns.set_theme(style="whitegrid", context="talk")
    ax = sns.barplot(x='Class', y='F1 Score', hue='Dataset', data=plot_df, palette='coolwarm')
    ax.set_title('Cross-Domain Generalization (F1) with Threshold Adaptation', fontsize=18, pad=15)
    ax.set_ylim(0, 1.05)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontsize=12)
    plt.legend(title='Dataset (threshold)', bbox_to_anchor=(1, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    print(f"\nSaved generalisation plot to {output_path}")
    plt.show()

    return plot_df

generalisation_results = cross_dataset_validation()
